In [2]:
import pandas as pd

Zakomentowane wczytywanie plików niepotrzebnych do merga z innym zbiorem, bo zajmuje więcej czasu (ok. 7 minut) - można odkomentować

In [ ]:
#names = pd.read_csv('../data/imdb/name.basics.tsv.gz', sep='\t')
#names.head()

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0117057"
2,nm0000003,Brigitte Bardot,1934,2025,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949,1982,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918,2007,"writer,director,actor","tt0050986,tt0069467,tt0050976,tt0083922"


Tu mamy informacje o osobach (nie tylko aktorach, ale ogólnie wszystkich osobach związanych z filmami)
- Imie i nazwisko
- Daty urodzin / śmierci (może przydatne jedynie do filtrowania po żyjących)
- professions (tutaj po przecinku wymienionych kilka, w dalszych analizach albo to trzeba robić na wiele wierszy, albo na wiele kolumn i zrobić one-hot-encoding)
- tytuły z którymi są związani (znów ta sama sytuacja że wiele po przecinku)

In [ ]:
#title_akas = pd.read_csv('../data/imdb/title.akas.tsv.gz', sep='\t')
#title_akas.head()

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle
0,tt0000001,1,Carmencita,\N,\N,original,\N,1
1,tt0000001,2,Carmencita,DE,\N,\N,literal title,0
2,tt0000001,3,Carmencita,US,\N,imdbDisplay,\N,0
3,tt0000001,4,Carmencita - spanyol tánc,HU,\N,imdbDisplay,\N,0
4,tt0000001,5,Καρμενσίτα,GR,\N,imdbDisplay,\N,0


Tutaj mamy wiele wersji tego samego filmu. Jest oryginalny tytuł i dzięki temu można wziąć region i język. Wszystkie inne duplikaty tego samego tytułu w różnych wersjach jezykowych trzeba by usunąc. Ale jest ryzykowne łączenie potem po tytule oryginalnym, bo w innych bazach będą to tytuły angielskie


In [23]:
title_basics = pd.read_csv('../data/imdb/title.basics.tsv.gz', sep='\t')
title_basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [ ]:
#title_crew = pd.read_csv('../data/imdb/title.crew.tsv.gz', sep='\t')
#title_crew.head()

,tconst,directors,writers
0,tt0000001,nm0005690,\N
1,tt0000002,nm0721526,\N
2,tt0000003,nm0721526,nm0721526
3,tt0000004,nm0721526,\N
4,tt0000005,nm0005690,\N


To jest dość ważne bo mamy od razu info o reżyserze i scenarzyście (dokładnie tego potrzebujemy do naszej analizy) i to trzeba będzie łączyć po id z tabelą names
UWAGA bo czasem jeden film ma kilku reżyserów/aktorów dlatego trzeba to uważnie połączyć

In [ ]:
# Przygotowanie reżyserów do łączenia
directors_exploded = title_crew[['tconst', 'directors']].copy()
directors_exploded['directors'] = directors_exploded['directors'].str.split(',')
directors_exploded = directors_exploded.explode('directors')
directors_exploded.head(10)

,tconst,directors
0,tt0000001,nm0005690
1,tt0000002,nm0721526
2,tt0000003,nm0721526
3,tt0000004,nm0721526
4,tt0000005,nm0005690
5,tt0000006,nm0005690
6,tt0000007,nm0005690
6,tt0000007,nm0374658
7,tt0000008,nm0005690
8,tt0000009,nm0085156


In [47]:
directors_named = pd.merge(
    directors_exploded, 
    names[['nconst', 'primaryName']], 
    left_on='directors', 
    right_on='nconst', 
    how='left')

directors_named.drop('nconst', inplace=True, axis=1)

In [48]:
directors_named.head(10)

,tconst,directors,primaryName
0,tt0000001,nm0005690,William K.L. Dickson
1,tt0000002,nm0721526,Émile Reynaud
2,tt0000003,nm0721526,Émile Reynaud
3,tt0000004,nm0721526,Émile Reynaud
4,tt0000005,nm0005690,William K.L. Dickson
5,tt0000006,nm0005690,William K.L. Dickson
6,tt0000007,nm0005690,William K.L. Dickson
7,tt0000007,nm0374658,William Heise
8,tt0000008,nm0005690,William K.L. Dickson
9,tt0000009,nm0085156,Alexander Black


In [51]:
# Przygotowanie scenarzystów do łączenia
writers_exploded = title_crew[['tconst', 'writers']].copy()
writers_exploded['writers'] = writers_exploded['writers'].str.split(',')
writers_exploded = writers_exploded.explode('writers')
writers_exploded.head(10)

,tconst,writers
0,tt0000001,\N
1,tt0000002,\N
2,tt0000003,nm0721526
3,tt0000004,\N
4,tt0000005,\N
5,tt0000006,\N
6,tt0000007,\N
7,tt0000008,\N
8,tt0000009,nm0085156
9,tt0000010,\N


In [54]:
writers_named = pd.merge(
    writers_exploded, 
    names[['nconst', 'primaryName']], 
    left_on='writers', 
    right_on='nconst', 
    how='left')

writers_named.drop('nconst', inplace=True, axis=1)

In [55]:
writers_named.head(10)

,tconst,writers,primaryName
0,tt0000001,\N,NaN
1,tt0000002,\N,NaN
2,tt0000003,nm0721526,Émile Reynaud
3,tt0000004,\N,NaN
4,tt0000005,\N,NaN
5,tt0000006,\N,NaN
6,tt0000007,\N,NaN
7,tt0000008,\N,NaN
8,tt0000009,nm0085156,Alexander Black
9,tt0000010,\N,NaN


In [56]:
directors_named.to_csv("../data/merged/directors_named.csv", index = False)
writers_named.to_csv("../data/merged/writers_named.csv", index = False)

In [ ]:
#title_episode = pd.read_csv('../data/imdb/title.episode.tsv.gz', sep='\t')
#title_episode.head()

,tconst,parentTconst,seasonNumber,episodeNumber
0,tt0031458,tt32857063,\N,\N
1,tt0041951,tt0041038,1,9
2,tt0042816,tt0989125,1,17
3,tt0042889,tt0989125,\N,\N
4,tt0043426,tt0040051,3,42


Episodes nie potrzebujemy bo to będzie do odcinków seriali itp, całkiem możemy tą tabele ominąć

In [ ]:
#needed_cols = ['tconst', 'nconst', 'category'] 

#title_principals = pd.read_csv('../data/imdb/title.principals.tsv.gz', sep='\t', usecols=needed_cols)

To może być potrzebne żeby wyfiltrować np samych aktorów, ale to jest tak ogromny plik że próbując to załadować brakło mi pamięci a potem wyrzuciło mi cały kernel. Nie wiem czy wam się uda. Aktorów można też z innych plików wziąć

In [9]:
title_ratings = pd.read_csv('../data/imdb/title.ratings.tsv.gz', sep='\t')
title_ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2198
1,tt0000002,5.5,310
2,tt0000003,6.5,2304
3,tt0000004,5.1,196
4,tt0000005,6.2,3035


Nie wiem czy ratings będą nam potrzebne (bo jak mamy określać sukces filmu w przyszłości to ratingu jeszcze nie znamy). Ale może się wykorzysta np do tego jak są oceniane filmy z danym aktorem/reżyserem (?)

Tu jest fajna tego baza bo od razu mamy średnią ocene wielu użytkowników, a nie każdego użytkownika osobno jak to jest w innych plikach

In [24]:
title_movies = title_basics[(title_basics['primaryTitle'].notna()) & (title_basics['titleType'] == 'movie')].copy()

In [25]:
title_movies['startYear'] = pd.to_numeric(title_movies['startYear'], errors='coerce')
title_movies = title_movies[(title_movies['startYear'] >= 1900) & (title_movies['startYear'] < 2026)]
title_movies.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
498,tt0000502,movie,Bohemios,Bohemios,0,1905.0,\N,100,\N
570,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906.0,\N,70,"Action,Adventure,Biography"
587,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907.0,\N,90,Drama
610,tt0000615,movie,Robbery Under Arms,Robbery Under Arms,0,1907.0,\N,\N,Drama
625,tt0000630,movie,Hamlet,Amleto,0,1908.0,\N,\N,Drama


In [ ]:
title_movies.shape
# ponad 600TYS filmów

(624222, 9)

In [27]:
title_movies[(title_movies['primaryTitle'] == "Avatar") & (title_movies['startYear'] == 2009)]

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
479801,tt0499549,movie,Avatar,Avatar,0,2009.0,\N,162,"Action,Adventure,Fantasy"


In [28]:
box_office = pd.read_csv("../data/worldwide-box-office/boxoffice.csv")
merged = pd.merge(
    title_movies,
    box_office,
    left_on=['originalTitle', 'startYear'],
    right_on=['title', 'year']
)

In [29]:
print(len(title_movies))
print(len(merged[merged['title'].notna()]))

624222
16824


Na 624 222 filmy pomiędzy rokiem 1900 a 2025 ze źródeł IMDB, 16 822 udało się dopasować do danych z Worldwide Box Office.

In [30]:
merged.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,title,domestic_million_usd,overseas_million_usd,world_million_usd,year
0,tt0003471,movie,Traffic in Souls,Traffic in Souls,0,1913.0,\N,88,"Crime,Drama",Traffic in Souls,1.0,NaN,1.0,1913.0
1,tt0003743,movie,The Call of the North,The Call of the North,0,1914.0,\N,\N,"Adventure,Drama",The Call of the North,0.1,NaN,0.1,1914.0
2,tt0004294,movie,The Man from Home,The Man from Home,0,1914.0,\N,\N,Drama,The Man from Home,0.1,NaN,0.1,1914.0
3,tt0004336,movie,The Million Dollar Mystery,The Million Dollar Mystery,0,1914.0,\N,\N,"Adventure,Mystery,Romance",The Million Dollar Mystery,3.3,NaN,3.3,1914.0
4,tt0004545,movie,Rose of the Rancho,Rose of the Rancho,0,1914.0,\N,50,"Action,Adventure,Drama",Rose of the Rancho,0.1,NaN,0.1,1914.0


In [32]:
merged.to_csv("../data//merged/title_movies_merged.csv", index=False)